# 07 — Chatting with the trained models

This notebook exercises the same `generate()` / `load_checkpoint()` API that powers the `ddm-chat` terminal REPL (`src/ddm/generate.py`). The device is auto-detected: CUDA when available, CPU otherwise.

In [1]:
from pathlib import Path

from ddm.generate import generate, load_checkpoint

CKPT_DIR = Path.cwd().parent / "checkpoints"

In [2]:
checkpoints = sorted(CKPT_DIR.glob("*_seed*.pt"))
print("found:", [p.name for p in checkpoints])

models = {}
for path in checkpoints:
    try:
        model, config, tok = load_checkpoint(str(path))
    except Exception as exc:
        print(f"skip {path.name}: {exc}")
        continue
    models[path.stem] = (model, config)

if not models:
    raise SystemExit("no checkpoints loaded")

print("loaded:", sorted(models))
device = next(next(iter(models.values()))[0].parameters()).device
print("device:", device)

found: ['bigram_seed0.pt', 'ddm_ablation_seed0.pt', 'ddm_ablation_seed1.pt', 'ddm_seed0.pt', 'ddm_seed1.pt', 'ngram_seed0.pt', 'transformer_seed0.pt']


loaded: ['bigram_seed0', 'ddm_ablation_seed0', 'ddm_ablation_seed1', 'ddm_seed0', 'ddm_seed1', 'ngram_seed0', 'transformer_seed0']
device: cpu


## 1. One-shot generation across models

The same prompt is continued by every trained model. DDM and the transformer see the whole prompt; bigram conditions on a single token and 3-gram on three, which is why they drift quickly.

In [3]:
prompt = "The theory of relativity was developed by"
for name, (model, config) in models.items():
    out = generate(model, tok, prompt, max_new_tokens=50, seed=7)
    print(f"\n--- {name} ---\n{prompt}{out.text}")


--- bigram_seed0 ---
The theory of relativity was developed by Network helicoptersSpe� badly areas formation glide________________________________________________________________ Feature Networkrahimν helicoptersFemaleorously Feature civilian dystop RB glide Grove streetScotSpeカasso Bloomberg Bloomberg________________________________________________________________ civilianasso smoke dystopassorahim RBaji meticulouslyScotomalカ badly civilianorously downstairsomal passers acquisitions logic


--- ddm_ablation_seed0 ---
The theory of relativity was developed byrentices Principle TECH kindergmelencersanged Putin JudicialencersATHPhil kinderg Stephan footwearindle Stephan fallingaimon CEOs however Stephan effectively authorizedindleencersrentices brighter Stephan PutinPhil bal TECHimpl Stephanencersrist kinderg bal fucking Putin falling footwear kindergrenticesanged fucking� adjacentanged



--- ddm_ablation_seed1 ---
The theory of relativity was developed by shrugged whirlwind archaeological taxis Island IslandVirgin grades overwhelming followers resembled Iro foodsón--+izons stabbed whirlwind UTC foods resembled formaló formalón information moving Island imposed moving ¯ UTC ¯ whirlwind information Iroónizonsó)] ISISishers overwhelming failed Hermesishers stabbedoeuv formal hardships



--- ddm_seed0 ---
The theory of relativity was developed by Lover evangelicals Yuk Uganda Jacob Area seriousOwnerdd fetch Elise Iroibus <@ Lover ancestorsScreen Lover Among deerScreen playing EliseContextosure Abram Engels roadmap Elise AMDclaimed EliseosureContextclaimed Iro 283ddclaimed Rud plain deer Parkinson forclaimed Iro Among unwelcomeContext Jacob



--- ddm_seed1 ---
The theory of relativity was developed by Knife vertex motionヴァ motion addition Advancedoven Likely bicycles NB iPads Contest additionoven Likely Enhanceilities apple legallyflfl SeasflRyan Enhanceoven motion iPads depended gren addition legally bicycles iPadsaction smartphonesforcekeeper bustía^{action legally Enhance dreams Likelyeredith bustcig



--- ngram_seed0 ---
The theory of relativity was developed byimeo dermat messy Jurassic racially perceptual monkeysIAcircle Awareness Awareness wildfireIA nodd Adv suff suff Awareness Collider wildfire marrow frustrated Al wildfireIA 1955 conceptions Bereourge Susan marrowinateincinn 1955 Bere Jurassic Lob Adv dermatkson mages perceptualcircleMAX nodd Awareness suff cardiovascular exemption Awareness



--- transformer_seed0 ---
The theory of relativity was developed by aliveGaza bruises eclipse irreversible AttE Bie Hunt Airlines Pistonsggyemer Airlines today eclipse Julio contradictoryFollowE Bie jihad MLA Manson irreversibleGaza Manson foes jihad bruises ZurGaza AirlinesGazaggyrex bruisesessential eclipseoddy Manson bruises Mek Zureping Att deaf they Mek foes


## 2. Sampling parameters

`temperature` sharpens or flattens the distribution; `top_k` keeps only the k most likely tokens. Same seed, same prompt, same model — only the sampler changes.

In [4]:
name = "ddm_seed0"
model, config = models[name]
base = "The castle stood on the hill"

for temp in (0.2, 0.8, 1.5):
    out = generate(model, tok, base, max_new_tokens=40, temperature=temp,
                   top_k=0, top_p=1.0, seed=7)
    print(f"\n--- temperature {temp} ---\n{base}{out.text}")

for top_k in (1, 10, 50):
    out = generate(model, tok, base, max_new_tokens=40, temperature=0.8,
                   top_k=top_k, top_p=1.0, seed=7)
    print(f"\n--- top_k {top_k} ---\n{base}{out.text}")


--- temperature 0.2 ---
The castle stood on the hill second tomato Veteranシosaursmetics freelMB minersernaut、 miners Warehousehao sd specify tattmeticsExt Plaint oversees unchanged 181 oversees rebCle ----metics smpowder distracting nighttimelucent minersシ probes definitely contestantsippy INFO



--- temperature 0.8 ---
The castle stood on the hillercise examplesgieActiv oursALS crank fueling Gideonernaut、 toll Warehouse inhibitsent Tutorial tatt Coordinator209Tumblr reign tense 181 Spitmony Dodd ---- ranked sm champjab Truthprotected ignoredKaritis definitely codes COMPLE embro



--- temperature 1.5 ---
The castle stood on the hill botched examplesgie FEC --------------------------------ALS crank fueling Gideonernaut、 toll Warehouse inhibitsent TutorialFAULT Coordinator 1900 Daytona breakup tense 181 Spitmony Dodd ---- rankedyoutube champjab Truthprotected ignoredKaritis definitely codes COMPLE embro



--- top_k 1 ---
The castle stood on the hill miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners miners



--- top_k 10 ---
The castle stood on the hill post post minersシシmetics freeluku minersjusticeuku miners AtlasVILLE Atlasmeticsシmeticsmetics postuku Atlas postVILLEjustice Atlas minersmeticsGREENjustice postシVILLE Atlas freelGREENjustice Atlas Atlas post



--- top_k 50 ---
The castle stood on the hill contingency reach Veteran rivalsonsagascar freelMB minersjusticeuku miners Atlashao finanhao prepar enjoys understood Sunni oversees Atlasauld overseesjustice AtlasStrameticsasonableRESULTS postシ rivalsRESULTS freel probes reasons2200 Atlas INFO


## 3. Long context: DDM segment memory

The story below is longer than the 128-token window both models were trained with. DDM keeps a per-layer segment memory of everything before the window, so content from the beginning of the story can still influence the continuation; the transformer must discard it.

In [5]:
story = (
    "The northern kingdom was founded by King Aldric in the year 812. "
    "His daughter, Queen Elara, expanded the borders to the sea. "
    "The royal library held three thousand volumes of history. "
    "After the great fire of 941, the library was rebuilt in stone. "
    "Aldric's great-grandson, King Bram, signed the treaty of the bay. "
    "The treaty ended the war with the southern states. "
    "Bram's advisor was a woman named Selma, who kept the royal seal. "
    "She hid the seal in the hollow of an old oak tree. "
    "In the winter of 977, the seal was stolen by a thief named Dorian."
)
follow_up = " The royal seal was hidden"

story_ids = tok(story, return_tensors="pt")["input_ids"][0]
print(f"story length: {len(story_ids)} tokens "
      f"(model window: {models['ddm_seed0'][1].max_seq_len})")

for name in ("ddm_seed0", "transformer_seed0"):
    if name not in models:
        print(f"{name} not available, skipping")
        continue
    model, cfg = models[name]
    out = generate(model, tok, story + follow_up, max_new_tokens=40,
                   temperature=0.8, seed=7)
    print(f"\n--- {name} ---\n{story + follow_up}{out.text}")

story length: 129 tokens (model window: 128)



--- ddm_seed0 ---
The northern kingdom was founded by King Aldric in the year 812. His daughter, Queen Elara, expanded the borders to the sea. The royal library held three thousand volumes of history. After the great fire of 941, the library was rebuilt in stone. Aldric's great-grandson, King Bram, signed the treaty of the bay. The treaty ended the war with the southern states. Bram's advisor was a woman named Selma, who kept the royal seal. She hid the seal in the hollow of an old oak tree. In the winter of 977, the seal was stolen by a thief named Dorian. The royal seal was hidden irritationMat Number 353 flags flagsMat fluct duly requested Scolvesarbagin duly Newport ll flagscl undergo NBA wooden irritation 353arb ll irritation Stur dulyCourtesy pushing ll cliff separatists Settlement SettlementolvesHandlerussionarb



--- transformer_seed0 ---
The northern kingdom was founded by King Aldric in the year 812. His daughter, Queen Elara, expanded the borders to the sea. The royal library held three thousand volumes of history. After the great fire of 941, the library was rebuilt in stone. Aldric's great-grandson, King Bram, signed the treaty of the bay. The treaty ended the war with the southern states. Bram's advisor was a woman named Selma, who kept the royal seal. She hid the seal in the hollow of an old oak tree. In the winter of 977, the seal was stolen by a thief named Dorian. The royal seal was hidden jargon exclusion� SeahawksONDONPlan Bosnia Civilization disadvantressing Men 315inter lug Catholicism botheredcssabetesinter Located Bosnia� bothered Codec sound sound Inquadiatorlichinter210 digsDNA Located sound jargon mainstream Corkerample bite


## 4. Interactive chat widget

A small ipywidgets panel. Each click runs `generate()` on the selected model with streaming output; the terminal `ddm-chat` REPL behaves the same way.

In [6]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None

if widgets is None:
    print("ipywidgets not installed - run: uv pip install ipywidgets")
else:
    model_menu = widgets.Dropdown(options=sorted(models), value="ddm_seed0",
                                  description="model")
    max_tok = widgets.IntSlider(value=80, min=8, max=256, step=8,
                                description="max tokens")
    temp = widgets.FloatSlider(value=0.8, min=0.0, max=2.0, step=0.05,
                               description="temperature")
    prompt_box = widgets.Textarea(placeholder="Type a prompt...",
                                  description="prompt",
                                  layout=widgets.Layout(width="100%",
                                                       height="90px"))
    run_btn = widgets.Button(description="Generate")
    out = widgets.Output(layout=widgets.Layout(width="100%", height="300px",
                                               overflow_y="auto"))

    def on_run(_):
        with out:
            out.clear_output()
            text = prompt_box.value.strip()
            if not text:
                print("(empty prompt)")
                return
            model, cfg = models[model_menu.value]
            print(f"> {text}\n")
            result = generate(
                model, tok, text,
                max_new_tokens=max_tok.value,
                temperature=temp.value,
                on_token=lambda t: print(t, end="", flush=True),
            )
            print(f"\n[{len(result.token_ids)} tokens, "
                  f"{result.tokens_per_sec:.1f} tok/s]")

    run_btn.on_click(on_run)
    display(widgets.VBox([model_menu, max_tok, temp, prompt_box, run_btn, out]))

## 5. Summary

| Model | Context per step | Memory across turns |
|---|---|---|
| ddm | last 128 tokens | segment memory carried |
| ddm_ablation | last 128 tokens | segment memory carried (fixed 1/k gate) |
| transformer | last 128 tokens | none |
| bigram | 1 token | none |
| ngram | last 3 tokens | none |